# ODI to Databricks Migration: TRG_DEP_Insert**Converted On:** 2024-07-30 12:00:00 UTC**Description:** This notebook migrates an ODI INSERT statement for the `TRG_DEP` table from Oracle to Databricks Spark SQL. It performs a full load insert into the target table from the `DEPARTMENTS` source table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type (e.g., FULL)")dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "Last Extract Timestamp")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS SELECT '${ETL_JOB_TYPE}' AS etl_job_type;CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS SELECT CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id;CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS SELECT CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid;CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS SELECT CAST('${ODI_SESS_NO}' AS STRING) AS odi_sess_no;CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS SELECT to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;

In [ ]:
display(spark.sql("""
SELECT
  (SELECT etl_job_type FROM v_etl_job_type) AS ETL_JOB_TYPE,
  (SELECT datasource_num_id FROM v_datasource_num_id) AS DATASOURCE_NUM_ID,
  (SELECT etl_proc_wid FROM v_etl_proc_wid) AS ETL_PROC_WID,
  (SELECT odi_sess_no FROM v_odi_sess_no) AS ODI_SESS_NO,
  (SELECT etl_last_extract_time FROM v_etl_last_extract_time) AS ETL_LAST_EXTRACT_TIME
"""))

## Main Data Load: TRG_DEP

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: (No SQL provided in source for this step)-- SCEN_TASK_NO in {20}: (No SQL provided in source for this step)-- SCEN_TASK_NO in {30}: (No SQL provided in source for this step)INSERT INTO workspace.hr.trg_dep  (    department_id ,    department_name ,    manager_id ,    location_id  )SELECT  departments.department_id ,  departments.department_name ,  departments.manager_id ,  departments.location_idFROM  workspace.hr.departments AS departments;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS trg_dep_record_count FROM workspace.hr.trg_dep;

## Conversion Notes & Manual Actions Required1.  **Schema Configuration:** Ensure `workspace.hr` schema exists in your Databricks environment and that the running user has appropriate permissions to `INSERT` into `workspace.hr.trg_dep` and `SELECT` from `workspace.hr.departments`.2.  **Table Existence:** The original ODI source only provided an `INSERT` statement, implying the target table `TRG_DEP` and source `DEPARTMENTS` already exist. No `CREATE TABLE` DDL was provided for `TRG_DEP` in the input. Ensure `workspace.hr.trg_dep` and `workspace.hr.departments` tables are created with compatible schema definitions (e.g., Oracle `NUMBER` maps to `BIGINT` or `DECIMAL`, `VARCHAR2` maps to `STRING`).3.  **Parameter Widgets:** Placeholder widgets have been created for common ODI parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`, `ETL_LAST_EXTRACT_TIME`). Adjust or remove these if they are not used in other parts of your overall ETL process or if this specific flow doesn't require them.4.  **`SCEN_TASK_NO` Mapping:** The `SCEN_TASK_NO` entries `{10}`, `{20}`, `{30}` from the original ODI file did not contain executable SQL and have been kept as comments in the nearest relevant SQL cell to preserve context.